# 测试集逐事件似然回归

使用同一批测试事件比较 ST-NKF 与 ETAS 的时间、空间 LL。图中保留密度着色、线性回归、Pearson R² 和 1:1 线。读取后先核对两个 CSV 的事件及切分，避免将不同事件逐行相减或回归。

改编自 [zhancx-edu/NeuralKernelFunction 的 plot regression.ipynb](https://github.com/zhancx-edu/NeuralKernelFunction/blob/febc15aefc316cf128ea217bbd60321c88fbaab1/plot%20regression.ipynb)，适配本仓库 PyTorch 输出。

在仓库根目录启动 Jupyter，先安装 `requirements-plotting.txt`。修改下面的参数单元格后，从上到下执行。结果保存到 `fig/`；本 Notebook 不启动训练。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "run_models.py").is_file():
    raise FileNotFoundError("请在 ST-NKF-pytorch 仓库根目录启动 Jupyter / nbconvert。")

from scipy import stats
from matplotlib.ticker import MaxNLocator

In [ ]:
DATASETS = ["SCEDC_20", "SCEDC_25", "SCEDC_30", "ComCat", "SaltonSea", "SanJac", "WHITE"]
CSV_DIR = ROOT / "csv"
FIG_DIR = ROOT / "fig"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300
SHOW_FIGURES = True

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.linewidth": 0.8})

In [ ]:
MODEL_TAGS = {"ST-NKF": "neural_neural_neural", "ETAS": "empirical_empirical_empirical"}

def load_pair(dataset):
    frames = {}
    identity = ["step", "time_days", "magnitude", "x", "y"]
    required = identity + ["logli_t", "logli_s"]
    for model, tag in MODEL_TAGS.items():
        path = CSV_DIR / f"{dataset}_{tag}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；请先分别运行该目录的 ETAS 和 ST-NKF。")
        data = pd.read_csv(path)
        missing = set(required) - set(data.columns)
        if missing:
            raise ValueError(f"{path}: 缺少列 {sorted(missing)}")
        data = data.loc[data["step"].isin(["train", "val", "test"])].reset_index(drop=True)
        if data.empty:
            raise ValueError(f"{path}: 没有 train / val / test 事件。")
        numeric = ["time_days", "magnitude", "x", "y", "logli_t", "logli_s"]
        if not np.isfinite(data[numeric].to_numpy(dtype=float)).all():
            raise ValueError(f"{path}: 事件或似然列包含非有限值。")
        order = data["step"].map({"train": 0, "val": 1, "test": 2}).to_numpy()
        if np.any(np.diff(order) < 0):
            raise ValueError(f"{path}: 事件未按 train、val、test 顺序排列。")
        frames[model] = data
    nkf, etas = frames["ST-NKF"], frames["ETAS"]
    if not nkf[identity].equals(etas[identity]):
        raise ValueError(f"{dataset}: 两个 CSV 的事件或切分不一致，不能逐行比较。")
    return nkf, etas

def save_figure(fig, stem, pdf=False):
    fig.savefig(FIG_DIR / (stem + ".png"), dpi=DPI, bbox_inches="tight")
    if pdf:
        fig.savefig(FIG_DIR / (stem + ".pdf"), bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)
    print("Saved:", FIG_DIR / (stem + ".png"))

def plot_regression(x, y, dataset, kind):
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        raise ValueError(f"{dataset} {kind}: 回归需要至少三个点且两轴不能为常量。")
    slope, intercept = np.polyfit(x, y, 1)
    r2 = stats.pearsonr(x, y)[0] ** 2
    fig, ax = plt.subplots(figsize=(3.8, 3.2))
    try:
        xy = np.vstack([x, y])
        density = stats.gaussian_kde(xy)(xy)
    except np.linalg.LinAlgError:
        # 完全线性关系时二维 KDE 奇异，仍绘制回归，但不伪造密度色条。
        ax.scatter(x, y, s=10, color="#1f77b4", alpha=0.8)
    else:
        order = density.argsort()
        scatter = ax.scatter(x[order], y[order], c=density[order] * 100, s=10,
                             cmap="RdBu_r", alpha=0.8, edgecolors="none")
        fig.colorbar(scatter, ax=ax, shrink=0.8, pad=0.02, label="Density")
    line_x = np.array([x.min(), x.max()])
    ax.plot(line_x, slope * line_x + intercept, color="red", lw=1.5,
            label=f"Regression, R²={r2:.4f}")
    limits = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(limits, limits, "--", color="#444444", lw=1.2, label="1:1 line")
    ax.set_xlabel(f"{kind.capitalize()} log-likelihood, ST-NKF")
    ax.set_ylabel(f"{kind.capitalize()} log-likelihood, ETAS")
    ax.set_title(dataset)
    ax.grid(True, ls="--", lw=0.3, alpha=0.5)
    ax.legend(frameon=False, fontsize=8)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    fig.tight_layout()
    save_figure(fig, f"regression-{dataset}_{kind}")

In [ ]:
for dataset in DATASETS:
    nkf, etas = load_pair(dataset)
    test = nkf["step"].eq("test")
    print(f"{dataset}: test={int(test.sum())}")
    for kind, column in (("temporal", "logli_t"), ("spatial", "logli_s")):
        plot_regression(nkf.loc[test, column].to_numpy(),
                        etas.loc[test, column].to_numpy(), dataset, kind)